# Incident assistant: CPU benchmark of quantized LLM, speech-to-text and text-to-speech

This notebook answers one question before the hackathon: **how good are the candidate models when
they are quantized?** Everything runs on CPU, and every model is a quantized build:

| Part | What is tested | Quantized build used here | What the phone would use |
|---|---|---|---|
| 1. LLM (form filling) | Qwen3-4B-Instruct-2507 vs Gemma-4-E4B-it | 4-bit GGUF (Q4_K_M) via llama.cpp | 4-bit (w4a16) builds from Qualcomm AI Hub |
| 2. Text-to-speech | Kokoro-82M, MeloTTS, Piper | int8 ONNX via sherpa-onnx | MeloTTS-EN from AI Hub |
| 3. Speech-to-text | Whisper tiny / base / small | int8 via faster-whisper | Whisper-Small-Quantized (w8a16) from AI Hub |
| 4. Full pipeline | speech → text → filled form | best of the above | — |

**How to run on Kaggle**
1. Settings → Accelerator: **None (CPU)**. Settings → **Internet: On** (needs a phone-verified account).
2. Optional: Add-ons → Secrets → add `HF_TOKEN` (a Hugging Face token). Needed only for the Svarah
   Indian-English speech set, after accepting its terms on huggingface.co/datasets/ai4bharat/Svarah.
3. For a first smoke run set `QUICK = True` (about 15–20 min). The full run takes roughly 1–1.5 hours on Kaggle's 4 CPU cores.
4. Run All. Results print at the end and are saved under `/kaggle/working/results/`.

**Read the speed numbers with care:** they are CPU numbers. On the phone the models run on the NPU and
will be faster. Accuracy carries over roughly, but the quantization formats differ slightly
(GGUF Q4_K_M here vs Qualcomm w4a16; CTranslate2 int8 here vs Qualcomm w8a16).

In [ ]:
# ==== Settings: change these, then Run All ====
import os, sys, re, json, time, math, glob, shutil, subprocess, tarfile, zipfile, io, random, difflib, platform
from pathlib import Path

QUICK = False          # True = only 6 test reports per test (smoke run). False = all 25.
RUN_LLM, RUN_TTS, RUN_ASR, RUN_PIPELINE = True, True, True, True

# LLMs: 4-bit GGUF files. The first repo + quant that exists is used.
LLM_CANDIDATES = {
    "Qwen3-4B-Instruct-2507": {"run": True, "repos": [
        "unsloth/Qwen3-4B-Instruct-2507-GGUF", "Qwen/Qwen3-4B-Instruct-2507-GGUF",
        "lmstudio-community/Qwen3-4B-Instruct-2507-GGUF", "bartowski/Qwen_Qwen3-4B-Instruct-2507-GGUF"]},
    "Gemma-4-E4B-it": {"run": True, "repos": [
        "ggml-org/gemma-4-E4B-it-GGUF", "unsloth/gemma-4-E4B-it-GGUF", "bartowski/google_gemma-4-E4B-it-GGUF"]},
    "Qwen3-1.7B (small fallback)": {"run": False, "repos": [
        "unsloth/Qwen3-1.7B-GGUF", "Qwen/Qwen3-1.7B-GGUF"]},
}
GGUF_QUANTS = ["Q4_K_M", "Q4_0", "UD-Q4_K_XL", "Q4_K_S", "IQ4_XS"]   # preference order
LLM_MODES = ["free"]   # "free" = plain JSON, like the phone. Add "schema" to also test grammar-forced JSON.
LLM_CTX = 4096         # AI Hub's LLM builds use a 4096-token context

# Speech-to-text: Whisper via faster-whisper, int8 weights.
ASR_MODELS = ["tiny", "base", "small"]
ASR_BEAM = 1           # greedy decoding, like a phone app
PIPELINE_ASR = "small"

# Text-to-speech: int8 ONNX voices from sherpa-onnx. First matching download per voice is used.
TTS_CANDIDATES = {
    "Kokoro-82M": ["kokoro-int8-en-v0_19", "kokoro-int8-multi-lang-v1_1", "kokoro-int8-multi-lang-v1_0", "kokoro-en-v0_19"],
    "MeloTTS (zh_en)": ["vits-melo-tts-zh_en"],
    "Piper lessac-medium": ["vits-piper-en_US-lessac-medium"],
}
NOISE_SNR_DB = 10      # background-chatter level for the "noisy hall" test (lower = noisier)

# Optional extra speech data
USE_SVARAH, SVARAH_N = True, 60   # Indian-accented English; needs HF_TOKEN secret + accepted terms
OWN_AUDIO_DIR = None   # e.g. "/kaggle/input/my-recordings": audio files + metadata.csv with columns file,text[,case_id][,lang]

THREADS = os.cpu_count() or 4
WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd() / "bench_work"
WORK.mkdir(parents=True, exist_ok=True)
OUT = WORK / "results"
OUT.mkdir(exist_ok=True)
random.seed(0)


def get_hf_token():
    tok = os.environ.get("HF_TOKEN")
    if tok:
        return tok
    try:
        from kaggle_secrets import UserSecretsClient
        tok = UserSecretsClient().get_secret("HF_TOKEN")
        os.environ["HF_TOKEN"] = tok
        return tok
    except Exception:
        return None


print(f"CPU threads: {THREADS} | Python {platform.python_version()} | HF token: {'yes' if get_hf_token() else 'no'} | output -> {OUT}")

In [ ]:
def pip_install(*pkgs):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], capture_output=True, text=True)
    if r.returncode:
        print(r.stderr[-2000:])


pip_install("faster-whisper", "jiwer", "sherpa-onnx", "onnxruntime", "onnx", "soundfile", "num2words",
            "huggingface_hub", "requests", "pandas", "scipy")

import numpy as np
import pandas as pd
import requests

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 60)


def download(url, dest, chunk=1 << 20):
    dest = Path(dest)
    tmp = dest.with_name(dest.name + ".part")
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(tmp, "wb") as f:
            for b in r.iter_content(chunk):
                f.write(b)
    tmp.rename(dest)
    return dest

## The test set

25 hand-written incident reports, phrased the way people actually talk: fillers, self-corrections
("11, no sorry, 11:30"), relative times ("half an hour ago"), three Hinglish reports, and four reports
with required details left out on purpose (to test whether the assistant knows it must ask a follow-up).
Each has the correct answer for every field. Relative times assume it is **Saturday 10 October 2026, 16:00**.

Some fields accept more than one answer (a fall with an injury can be "Slip/Trip/Fall" or "Personal Injury";
severity accepts the neighbouring level where it is a judgement call). Free-text fields are checked by key words.

In [ ]:
NOW_TEXT = "Saturday, 10 October 2026, 16:00"
TODAY, YESTERDAY = "2026-10-10", "2026-10-09"
TYPES = ["Fire/Smoke", "Water Leak/Flooding", "Electrical Hazard", "Personal Injury", "Slip/Trip/Fall",
         "Equipment Failure", "Chemical/Gas Leak", "Security/Theft", "Vehicle Incident", "Near Miss", "Other"]
SEVERITIES = ["Low", "Medium", "High", "Critical"]
MANDATORY = ["location", "date", "time"]   # type, severity and description are always filled by the model


def case(id, text, types, severity, location, date, time, equipment=None, actions=None, desc=(),
         persons=(), witnesses=(), people_any=(), lang="en", time_tol=0, optional=()):
    """location / equipment / actions: None = must be left empty; list = any of these key phrases must appear.
    severity: first entry is the main answer, others are also accepted.
    persons / witnesses: names that must be in that list; people_any: names that may be in either list.
    optional: fields where an empty answer is also fine."""
    return dict(id=id, text=text, lang=lang, type=list(types), severity=list(severity), location=location,
                date=date, time=time, time_tol=time_tol, equipment=equipment, actions=actions, desc=list(desc),
                persons=list(persons), witnesses=list(witnesses), people_any=list(people_any), optional=set(optional))


TEST_CASES = [
    case("c01", "There was a water leak near Conference Room 2 around 10 AM. Maintenance has already isolated the area.",
         ["Water Leak/Flooding"], ["Low", "Medium"], ["conference room 2"], TODAY, "10:00",
         actions=["isolat"], desc=["leak"]),
    case("c02", "Umm so, there was smoke coming out of the microwave in the third floor pantry, this was like quarter past one. "
                "Priya from HR used the fire extinguisher and it's out now. Nobody got hurt.",
         ["Fire/Smoke"], ["Low", "Medium"], ["pantry"], TODAY, "13:15",
         equipment=["microwave"], actions=["extinguish"], desc=["smoke", "fire"], people_any=["Priya"]),
    case("c03", "Hi, this is about Suresh from the warehouse team. He slipped on some oil near loading dock B at about 8:40 "
                "this morning and hurt his wrist. We took him to the first aid room. His supervisor Kavita saw the whole thing.",
         ["Slip/Trip/Fall", "Personal Injury"], ["Medium", "High"], ["dock b"], TODAY, "08:40",
         actions=["first aid"], desc=["slip", "fell", "fall"], persons=["Suresh"], witnesses=["Kavita"]),
    case("c04", "Just now a server rack started giving off a burning smell. I've cut the power to it.",
         ["Electrical Hazard", "Equipment Failure", "Fire/Smoke"], ["Medium", "High"], None, TODAY, "16:00",
         time_tol=10, equipment=["server rack", "rack", "server"], actions=["power", "cut", "switch"], desc=["burn", "smell"]),
    case("c05", "The forklift hit a shelf in aisle 7 at, uh, 11, no sorry, 11:30. Nobody was hurt but a few boxes fell down. "
                "The driver was Manoj.",
         ["Vehicle Incident", "Other", "Equipment Failure"], ["Low", "Medium"], ["aisle 7"], TODAY, "11:30",
         equipment=["forklift"], desc=["forklift"], persons=["Manoj"]),
    case("c06", "Yesterday evening around 7, someone broke the glass door at the side entrance and a laptop is missing from "
                "reception. Security guard Ramesh noticed it during his round. We have informed the police.",
         ["Security/Theft"], ["Medium", "High"], ["side entrance", "reception"], YESTERDAY, "19:00",
         equipment=["laptop", "door", "glass"], actions=["police"], desc=["laptop", "theft", "stolen", "missing", "broke"],
         people_any=["Ramesh"]),
    case("c07", "In chemistry lab 204, a bottle of hydrochloric acid fell and spilled on the floor at around 2:15 pm. "
                "Two students, Arjun and Meera, got some on their shoes but there was no skin contact. The lab assistant "
                "neutralised it with baking soda and we opened all the windows.",
         ["Chemical/Gas Leak"], ["Medium", "Low"], ["204"], TODAY, "14:15",
         equipment=["acid", "bottle"], actions=["neutralis", "neutraliz", "baking soda", "window"], desc=["acid", "spill"],
         persons=["Arjun", "Meera"]),
    case("c08", "This isn't really an accident, but a ceiling panel fell in the cafeteria right next to where people were "
                "sitting. It missed everyone. Happened around 12:50.",
         ["Near Miss", "Other"], ["Low", "Medium"], ["cafeteria"], TODAY, "12:50",
         equipment=["panel", "ceiling"], desc=["ceiling", "panel"]),
    case("c09", "Bhai, ground floor ke washroom mein pipe burst ho gaya hai, paani corridor tak aa gaya hai. "
                "Subah 9 baje ke aas paas hua. Plumber ko bula liya hai.",
         ["Water Leak/Flooding"], ["Medium", "Low"], ["washroom"], TODAY, "09:00",
         equipment=["pipe"], actions=["plumber"], desc=["pipe", "water", "burst", "leak"], lang="hinglish"),
    case("c10", "Generator room mein aag lag gayi thi around 3:30, fire brigade aayi thi, ab control mein hai. "
                "Ek technician Imran ke haath pe burn hua hai, usko hospital le gaye.",
         ["Fire/Smoke"], ["High", "Critical"], ["generator room"], TODAY, "15:30",
         equipment=["generator"], actions=["fire brigade", "hospital"], desc=["fire"], persons=["Imran"], lang="hinglish"),
    case("c11", "Someone collapsed and hit their head, there's blood on the floor. We've called an ambulance.",
         ["Personal Injury", "Slip/Trip/Fall", "Other"], ["High", "Critical"], None, None, None,
         actions=["ambulance"], desc=["collaps", "head", "blood"]),
    case("c12", "Sparks were coming from the switchboard near the main lobby reception desk at about 10:20 today. "
                "Our electrician Dinesh switched off the main breaker. Nobody was hurt.",
         ["Electrical Hazard"], ["Low", "Medium"], ["lobby"], TODAY, "10:20",
         equipment=["switchboard"], actions=["breaker", "switch"], desc=["spark"], people_any=["Dinesh"]),
    case("c13", "There's a strong smell of gas in the kitchen of hostel block C. I think the LPG cylinder is leaking. "
                "We noticed it at 6:45 this morning, evacuated the kitchen staff and closed the valve.",
         ["Chemical/Gas Leak"], ["High", "Critical"], ["kitchen"], TODAY, "06:45",
         equipment=["cylinder", "lpg"], actions=["evacuat", "valve"], desc=["gas"]),
    case("c14", "Lift number 3 in Tower B got stuck between the fifth and sixth floor at 9:10 am with four people inside. "
                "Maintenance got them out after twenty minutes and they are all fine.",
         ["Equipment Failure"], ["Medium", "Low"], ["tower b"], TODAY, "09:10",
         equipment=["lift", "elevator"], actions=["maintenance", "rescu", "got them out", "freed", "evacuat", "releas"],
         desc=["lift", "elevator", "stuck"]),
    case("c15", "Okay so I was walking to the basement parking, P2, and I saw one of the cars had its hazard lights on and "
                "there was oil leaking under it, quite a lot actually. Then the security guy, I think his name is Venkat, came "
                "and put sand on it. This was maybe half an hour ago. I don't think anyone slipped or anything.",
         ["Chemical/Gas Leak", "Vehicle Incident", "Other", "Near Miss"], ["Low", "Medium"], ["p2", "basement", "parking"],
         TODAY, "15:30", time_tol=15, equipment=["car", "vehicle"], actions=["sand"], desc=["oil"], people_any=["Venkat"]),
    case("c16", "I want to report a broken window.",
         ["Other", "Security/Theft", "Equipment Failure"], ["Low", "Medium"], None, None, None,
         equipment=["window"], desc=["window"], optional={"equipment"}),
    case("c17", "At 07:30 hours on 9th October, contractor employee Rajesh Kumar got a cut on his left hand while operating "
                "the bench grinder in the maintenance workshop. First aid was given and he was sent to hospital for stitches. "
                "Witness: Anil Sharma.",
         ["Personal Injury"], ["High", "Medium"], ["workshop"], YESTERDAY, "07:30",
         equipment=["grinder"], actions=["first aid", "hospital"], desc=["cut", "hand", "grinder"],
         persons=["Rajesh"], witnesses=["Anil"]),
    case("c18", "Water from the AC unit dripped onto the printer in the finance department around noon and now the printer "
                "won't turn on. I've put a bucket under the AC.",
         ["Water Leak/Flooding", "Equipment Failure"], ["Low", "Medium"], ["finance"], TODAY, "12:00",
         equipment=["printer", "ac", "air condition"], actions=["bucket"], desc=["printer", "water", "drip"]),
    case("c19", "There's a snake near the cycle stand behind the library. We spotted it about five minutes ago, "
                "cordoned off the area and called campus security.",
         ["Other", "Near Miss"], ["Medium", "Low"], ["cycle stand", "library"], TODAY, "15:55", time_tol=10,
         actions=["cordon", "security"], desc=["snake"]),
    case("c20", "The projector in conference room 12 caught fire at 3 pm. Neha unplugged it and threw water on it, "
                "no wait, she used the CO2 extinguisher.",
         ["Fire/Smoke", "Electrical Hazard"], ["Medium", "Low"], ["conference room 12"], TODAY, "15:00",
         equipment=["projector"], actions=["extinguish", "co2", "unplug"], desc=["projector", "fire"], people_any=["Neha"]),
    case("c21", "Campus gate ke paas main road pe ek bike aur auto ki takkar ho gayi. Bike wala ladka gir gaya, uske pair "
                "mein chot aayi hai. Ye abhi 10 minute pehle hua. Guard ne ambulance bula li hai.",
         ["Vehicle Incident"], ["High", "Medium"], ["gate", "main road"], TODAY, "15:50", time_tol=10,
         equipment=["bike", "auto", "motorcycle", "rickshaw"], actions=["ambulance"],
         desc=["bike", "auto", "accident", "collision", "collided", "crash"], lang="hinglish"),
    case("c22", "Major fire in the chemical store room of B block, flames are visible from outside. The whole building is "
                "being evacuated and the fire brigade has been called. It started about 5 minutes ago.",
         ["Fire/Smoke"], ["Critical", "High"], ["store"], TODAY, "15:55", time_tol=10,
         equipment=["chemical"], actions=["evacuat", "fire brigade"], desc=["fire"], optional={"equipment"}),
    case("c23", "My colleague Deepa tripped over a loose cable at her desk in the open office on the second floor around "
                "11:15. She's okay, just a bruise on her knee. Rohit and I saw it happen.",
         ["Slip/Trip/Fall", "Personal Injury"], ["Low", "Medium"], ["second floor", "open office", "2nd floor"], TODAY, "11:15",
         equipment=["cable"], desc=["trip", "cable"], persons=["Deepa"], witnesses=["Rohit"], optional={"equipment"}),
    case("c24", "Compressor C-12 in plant 2 tripped with a loud bang at 5:05 am. The night shift operator Sanjay shut it "
                "down and tagged it out.",
         ["Equipment Failure"], ["Medium", "Low"], ["plant 2"], TODAY, "05:05",
         equipment=["compressor"], actions=["shut", "tag"], desc=["compressor"], people_any=["Sanjay"]),
    case("c25", "The emergency exit door on the fourth floor is jammed and won't open, and someone has put chairs in front "
                "of it as well.",
         ["Equipment Failure", "Other"], ["High", "Medium"], ["fourth floor", "4th floor"], None, None,
         equipment=["door", "exit"], desc=["exit", "door"]),
]
CASE_BY_ID = {c["id"]: c for c in TEST_CASES}
QUICK_IDS = ["c01", "c03", "c04", "c09", "c11", "c20"]
CASES = [CASE_BY_ID[i] for i in QUICK_IDS] if QUICK else TEST_CASES
EN_CASES = [c for c in CASES if c["lang"] == "en"]
pd.DataFrame([{"id": c["id"], "lang": c["lang"], "missing on purpose": ", ".join(f for f in MANDATORY if c[f] is None) or "-",
               "text": c["text"]} for c in CASES])

In [ ]:
EXPECTED_KEYS = ["incident_type", "severity", "location", "date", "time", "persons_involved", "equipment",
                 "description", "immediate_actions", "witnesses", "follow_up"]
KEY_ALIASES = {"type": "incident_type", "incidenttype": "incident_type", "severity_level": "severity",
               "incident_location": "location", "date_time": "date", "persons": "persons_involved",
               "people_involved": "persons_involved", "equipment_involved": "equipment", "asset": "equipment",
               "equipment_asset": "equipment", "incident_description": "description",
               "immediate_actions_taken": "immediate_actions", "actions_taken": "immediate_actions",
               "witness": "witnesses", "witness_information": "witnesses", "recommended_follow_up": "follow_up",
               "follow_up_action": "follow_up", "recommended_follow_up_action": "follow_up"}
GENERIC = set("""someone somebody unknown unidentified person persons people employee employees worker workers staff member
members driver student students guard guards security technician operator colleague colleagues i me myself we us reporter
caller maintenance team mr mrs ms dr sir madam bhai ji the a an and of from in on at with his her their our my lab assistant
supervisor electrician plumber contractor night shift rider boy man woman girl victim injured kitchen four three two one five
six several few many all none na hr warehouse campus fire brigade ambulance police bike auto ladka wala inside public visitor
visitors other others not mentioned name named who whose crew department intruder intruders thief thieves culprit
suspect suspects occupants occupant passengers passenger individuals individual self unnamed patient pedestrian
nobody no one""".split())
EMPTY_WORDS = {"", "null", "none", "n a", "na", "nil", "unknown", "not mentioned", "not specified", "not provided",
               "not stated", "unspecified", "not given", "not available"}


def norm(s):
    s = str(s).lower().replace("a.m.", "am").replace("p.m.", "pm")
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    return re.sub(r"\s+", " ", s).strip()


def as_text(v):
    if v is None:
        return ""
    if isinstance(v, (list, tuple)):
        return "; ".join(as_text(x) for x in v)
    if isinstance(v, dict):
        return " ".join(f"{k} {as_text(x)}" for k, x in v.items())
    return str(v)


def is_empty(v):
    if v is None:
        return True
    if isinstance(v, (list, tuple)):
        return all(is_empty(x) for x in v)
    if isinstance(v, dict):
        return all(is_empty(x) for x in v.values())
    return norm(v) in EMPTY_WORDS


def key_match(pred, alts):
    """True if, for any alternative phrase, every word of it appears in the prediction.
    Words of 1-2 characters and numbers must match a whole word; longer words match as a word prefix."""
    toks = norm(as_text(pred)).split()
    for alt in alts:
        ok = True
        for w in norm(alt).split():
            hit = (w in toks) if (w.isdigit() or len(w) <= 2) else any(t.startswith(w) for t in toks)
            if not hit:
                ok = False
                break
        if ok:
            return True
    return False


def parse_date(v):
    if is_empty(v):
        return None
    s = as_text(v)
    m = re.search(r"(\d{4})-(\d{1,2})-(\d{1,2})", s)
    if m:
        return f"{int(m.group(1)):04d}-{int(m.group(2)):02d}-{int(m.group(3)):02d}"
    m = re.search(r"(\d{1,2})[/.](\d{1,2})[/.](\d{4})", s)
    if m:
        return f"{int(m.group(3)):04d}-{int(m.group(2)):02d}-{int(m.group(1)):02d}"
    return "unparsed:" + s


def parse_time(v):
    """Minutes since midnight, None if empty, -1 if unreadable."""
    if is_empty(v):
        return None
    s = as_text(v).lower().replace("a.m.", "am").replace("p.m.", "pm")
    s = re.sub(r"\d{4}-\d{1,2}-\d{1,2}[t ]?", "", s)
    if "noon" in s and not re.search(r"\d", s):
        return 12 * 60
    m = re.search(r"(\d{1,2})(?:[:.](\d{2}))?(?::\d{2})?\s*(am|pm)?", s)
    if not m:
        return -1
    h, mi, ap = int(m.group(1)), int(m.group(2) or 0), m.group(3)
    if ap == "pm" and h < 12:
        h += 12
    if ap == "am" and h == 12:
        h = 0
    return h * 60 + mi


def hm(t):
    h, m = t.split(":")
    return int(h) * 60 + int(m)


def person_entries(v):
    if v is None:
        return []
    if isinstance(v, str):
        v = [] if is_empty(v) else re.split(r",|;|\band\b", v)
    if isinstance(v, dict):
        v = list(v.values())
    out = []
    for x in v:
        if isinstance(x, dict):
            x = " ".join(str(y) for y in x.values())
        toks = [t for t in norm(x).split() if t not in GENERIC and len(t) > 1 and not t.isdigit()]
        if toks:
            out.append(toks)
    return out


def _similar(a, b):
    return a == b or difflib.SequenceMatcher(None, a, b).ratio() >= 0.8


def name_hit(name, entries):
    n = norm(name).split()[0]
    return any(_similar(n, t) for e in entries for t in e)


def parse_output(raw):
    """Returns (dict or None, clean, keys_ok). clean = the reply was only JSON, nothing else."""
    t = re.sub(r"<think>.*?</think>", "", raw or "", flags=re.S).strip()
    t2 = re.sub(r"^```(?:json)?\s*", "", t)
    t2 = re.sub(r"\s*```\s*$", "", t2).strip()
    obj, clean = None, False
    try:
        obj = json.loads(t2)
        clean = t2 == t
    except Exception:
        m = re.search(r"\{.*\}", t, flags=re.S)
        if m:
            try:
                obj = json.loads(m.group(0))
            except Exception:
                obj = None
    if not isinstance(obj, dict):
        return None, False, False
    keys_ok = set(obj.keys()) == set(EXPECTED_KEYS)
    out = {}
    for k, v in obj.items():
        kk = re.sub(r"[\s\-]+", "_", str(k).strip().lower())
        out[KEY_ALIASES.get(kk, kk)] = v
    return out, clean, keys_ok


FIELD_COLS = ["type_ok", "severity_ok", "location_ok", "date_ok", "time_ok", "equipment_ok", "actions_ok", "desc_ok"]


def score_case(c, raw):
    pred, clean, keys_ok = parse_output(raw)
    gold_people = c["persons"] + c["witnesses"] + c["people_any"]
    s = dict(json_valid=pred is not None, json_clean=clean, schema_ok=False)
    if pred is None:
        s.update({k: False for k in FIELD_COLS + ["severity_within1", "followup_ok", "missing_ok"]})
        s.update(people_recall=(0.0 if gold_people else None), invented_people=0, halluc=0, empty_slots=0,
                 fields_ok=0, fields_n=len(FIELD_COLS), pred_json=None)
        return s
    ty, sev = as_text(pred.get("incident_type")), as_text(pred.get("severity"))
    s["schema_ok"] = bool(keys_ok and ty in TYPES and sev in SEVERITIES)
    s["type_ok"] = norm(ty) in {norm(t) for t in c["type"]}
    sev_idx = {norm(x): i for i, x in enumerate(SEVERITIES)}
    si = sev_idx.get(norm(sev))
    s["severity_ok"] = si is not None and SEVERITIES[si] in c["severity"]
    s["severity_within1"] = si is not None and abs(si - SEVERITIES.index(c["severity"][0])) <= 1

    halluc = empty_slots = 0

    def text_field(name, gold, val):
        nonlocal halluc, empty_slots
        empty = is_empty(val)
        if gold is None:
            if name in c["optional"]:
                return True
            empty_slots += 1
            halluc += 0 if empty else 1
            return empty
        if empty:
            return name in c["optional"]
        return key_match(val, gold)

    s["location_ok"] = text_field("location", c["location"], pred.get("location"))
    s["equipment_ok"] = text_field("equipment", c["equipment"], pred.get("equipment"))
    s["actions_ok"] = text_field("actions", c["actions"], pred.get("immediate_actions"))

    pd_ = parse_date(pred.get("date"))
    if c["date"] is None:
        empty_slots += 1
        halluc += 0 if pd_ is None else 1
        s["date_ok"] = pd_ is None
    else:
        s["date_ok"] = pd_ == c["date"]

    pt = parse_time(pred.get("time"))
    if c["time"] is None:
        empty_slots += 1
        halluc += 0 if pt is None else 1
        s["time_ok"] = pt is None
    else:
        s["time_ok"] = pt is not None and pt >= 0 and abs(pt - hm(c["time"])) <= c["time_tol"]

    s["desc_ok"] = key_match(pred.get("description"), c["desc"]) if c["desc"] else not is_empty(pred.get("description"))
    s["followup_ok"] = not is_empty(pred.get("follow_up"))

    P, W = person_entries(pred.get("persons_involved")), person_entries(pred.get("witnesses"))
    checks = [(n, P) for n in c["persons"]] + [(n, W) for n in c["witnesses"]] + [(n, P + W) for n in c["people_any"]]
    s["people_recall"] = (sum(name_hit(n, L) for n, L in checks) / len(checks)) if checks else None
    src = set(norm(c["text"]).split())
    s["invented_people"] = sum(1 for e in P + W if not any(_similar(t, u) for t in e for u in src))

    pred_missing = {f for f in MANDATORY if is_empty(pred.get(f))}
    gold_missing = {f for f in MANDATORY if c[f] is None}
    s["missing_ok"] = pred_missing == gold_missing
    s.update(halluc=halluc, empty_slots=empty_slots, fields_ok=sum(bool(s[k]) for k in FIELD_COLS),
             fields_n=len(FIELD_COLS), pred_json=json.dumps(pred, ensure_ascii=False))
    return s


def summarize_llm(df):
    rows = []
    for (label, mode), d in df.groupby(["label", "mode"], sort=False):
        pr = d["people_recall"].dropna()
        rows.append({
            "model": label, "mode": mode, "reports": len(d),
            "field accuracy %": 100 * d[FIELD_COLS].astype(float).mean(axis=1).mean(),
            "valid JSON %": 100 * d["json_valid"].mean(), "clean JSON %": 100 * d["json_clean"].mean(),
            "exact keys+options %": 100 * d["schema_ok"].mean(),
            **{f"{k[:-3]} %": 100 * d[k].astype(float).mean() for k in FIELD_COLS},
            "severity ±1 %": 100 * d["severity_within1"].astype(float).mean(),
            "people found %": 100 * pr.mean() if len(pr) else float("nan"),
            "invented people": int(d["invented_people"].sum()),
            "made-up values %": 100 * d["halluc"].sum() / max(d["empty_slots"].sum(), 1),
            "right follow-up decision %": 100 * d["missing_ok"].astype(float).mean(),
            "sec / report": d["wall_s"].astype(float).mean(),
            "gen tokens/s": d["gen_tps"].astype(float).mean() if "gen_tps" in d else float("nan"),
        })
    return pd.DataFrame(rows).round(1)

In [ ]:
# Sanity check of the scorer: perfect answers must score 100%, and planted mistakes must be caught.
def perfect_reply(c):
    first = lambda g: g[0] if g else None
    return json.dumps({
        "incident_type": c["type"][0], "severity": c["severity"][0], "location": first(c["location"]),
        "date": c["date"], "time": c["time"], "persons_involved": c["persons"] + c["people_any"],
        "equipment": first(c["equipment"]), "description": " ".join(c["desc"]) or "incident",
        "immediate_actions": first(c["actions"]), "witnesses": c["witnesses"], "follow_up": "Inspect and fix the cause."})


for c in TEST_CASES:
    s = score_case(c, perfect_reply(c))
    bad = [k for k in FIELD_COLS + ["missing_ok", "followup_ok", "json_clean", "schema_ok"] if not s[k]]
    assert not bad and s["halluc"] == 0 and s["invented_people"] == 0 and s["people_recall"] in (None, 1.0), (c["id"], bad, s)

c = CASE_BY_ID["c03"]
wrong = json.loads(perfect_reply(c))
wrong.update(time="9:40", witnesses=["Rahul"], persons_involved=["Kavita"])
s = score_case(c, "Here is the form:\n```json\n" + json.dumps(wrong) + "\n```")
assert s["json_valid"] and not s["json_clean"] and not s["time_ok"] and s["invented_people"] == 1 and s["people_recall"] == 0.0
c = CASE_BY_ID["c04"]
wrong = json.loads(perfect_reply(c))
wrong["location"] = "Server room"
s = score_case(c, json.dumps(wrong))
assert not s["location_ok"] and s["halluc"] == 1 and not s["missing_ok"]
assert score_case(c, "Sorry, I can't help with that.")["json_valid"] is False
print("Scorer self-check passed.")

## Part 1: LLM form filling (4-bit GGUF on CPU via llama.cpp)

Each model gets the same instructions and has to return one JSON object with the form fields.
Nothing forces the JSON format (mode `free`), which is what you will get on the phone unless your runtime
supports grammar-constrained output. Add `"schema"` to `LLM_MODES` to also see how much forcing the format helps.

What the columns mean:
* **field accuracy %**: average share of the 8 scored fields filled correctly (type, severity, location, date, time, equipment, actions, description).
* **valid JSON %** / **clean JSON %**: the reply could be read as the form / the reply was *only* the form, nothing else.
* **made-up values %**: of all the fields the speaker never mentioned, how often the model invented something.
* **invented people**: names that never appeared in what was said.
* **right follow-up decision %**: the model left exactly the right required fields (location, date, time) empty, so the app would ask exactly the right follow-up question.
* **sec / report** and **gen tokens/s**: CPU speed (the phone NPU will differ).

In [ ]:
def _runs(exe, env):
    try:
        r = subprocess.run([str(exe), "--version"], env=env, capture_output=True, text=True, timeout=120)
        return r.returncode == 0, (r.stdout + r.stderr)[-400:]
    except Exception as e:
        return False, repr(e)


def get_llama_server():
    """Prebuilt CPU llama.cpp from GitHub releases; builds from source if that fails."""
    bin_dir = WORK / "llama_bin"
    try:
        rel = requests.get("https://api.github.com/repos/ggml-org/llama.cpp/releases/latest", timeout=60).json()
        skip = ("vulkan", "cuda", "rocm", "hip", "sycl", "openvino", "musa", "arm64", "aarch64", "s390", "kleidi")
        assets = [a for a in rel.get("assets", []) if "ubuntu" in a["name"] and "x64" in a["name"]
                  and a["name"].endswith((".zip", ".tar.gz")) and not any(k in a["name"].lower() for k in skip)]
        if assets:
            a = sorted(assets, key=lambda a: len(a["name"]))[0]
            arch = WORK / a["name"]
            if not arch.exists():
                download(a["browser_download_url"], arch)
            shutil.rmtree(bin_dir, ignore_errors=True)
            bin_dir.mkdir()
            if a["name"].endswith(".zip"):
                zipfile.ZipFile(arch).extractall(bin_dir)
            else:
                with tarfile.open(arch) as tf:
                    tf.extractall(bin_dir)
            exe = next((p for p in bin_dir.rglob("llama-server") if p.is_file()), None)
            if exe:
                for p in exe.parent.iterdir():
                    if p.is_file():
                        p.chmod(0o755)
                env = dict(os.environ, LD_LIBRARY_PATH=f"{exe.parent}:{os.environ.get('LD_LIBRARY_PATH', '')}")
                ok, msg = _runs(exe, env)
                print(f"llama.cpp {rel.get('tag_name')} ({a['name']}):", "ready" if ok else "won't run -> " + msg)
                if ok:
                    return exe, env
        else:
            print("No Ubuntu x64 CPU build in the latest release; building from source.")
    except Exception as e:
        print("Prebuilt llama.cpp failed:", repr(e)[:300])
    src = WORK / "llama.cpp"
    if not src.exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp", str(src)], check=True)
    subprocess.run(["cmake", "-S", str(src), "-B", str(src / "build"), "-DCMAKE_BUILD_TYPE=Release",
                    "-DLLAMA_CURL=OFF", "-DGGML_NATIVE=ON", "-DLLAMA_BUILD_TESTS=OFF"], check=True)
    subprocess.run(["cmake", "--build", str(src / "build"), "-j", str(THREADS), "--target", "llama-server"], check=True)
    exe, env = src / "build" / "bin" / "llama-server", dict(os.environ)
    ok, msg = _runs(exe, env)
    print("llama.cpp built from source:", "ready" if ok else msg)
    return exe, env


def find_gguf(repos, quants):
    from huggingface_hub import list_repo_files
    for repo in repos:
        try:
            files = list_repo_files(repo)
        except Exception as e:
            print(f"  {repo}: not available ({type(e).__name__})")
            continue
        ggufs = [f for f in files if f.endswith(".gguf") and "mmproj" not in f.lower()]
        for q in quants:
            hits = [f for f in ggufs if q.lower() in f.lower()]
            if hits:
                return repo, sorted(hits, key=len)[0]
    return None, None


class LlamaServer:
    def __init__(self, exe, env, model_path, port=8089):
        self.exe, self.env, self.model, self.port = exe, env, model_path, port
        self.url = f"http://127.0.0.1:{port}"
        self.proc, self.load_s = None, None
        self.log = OUT / f"llama_server_{Path(model_path).stem}.log"

    def start(self, timeout=900):
        for extra in (["--reasoning-budget", "0"], []):
            cmd = [str(self.exe), "-m", str(self.model), "-c", str(LLM_CTX), "-t", str(THREADS),
                   "--host", "127.0.0.1", "--port", str(self.port), "--jinja", "-np", "1", *extra]
            self._logf = open(self.log, "w")
            self.proc = subprocess.Popen(cmd, env=self.env, stdout=self._logf, stderr=subprocess.STDOUT)
            t0 = time.time()
            while time.time() - t0 < timeout and self.proc.poll() is None:
                try:
                    if requests.get(self.url + "/health", timeout=5).status_code == 200:
                        self.load_s = time.time() - t0
                        return self
                except requests.RequestException:
                    pass
                time.sleep(2)
            self.stop()
            print("llama-server did not start with flags", extra or "(none)", "\n", self.log.read_text()[-1500:])
        raise RuntimeError(f"llama-server failed to start; see {self.log}")

    def stop(self):
        if self.proc and self.proc.poll() is None:
            self.proc.terminate()
            try:
                self.proc.wait(30)
            except subprocess.TimeoutExpired:
                self.proc.kill()
        self.proc = None
        if getattr(self, "_logf", None):
            self._logf.close()

    def chat(self, messages, schema=None, max_tokens=512):
        body = {"messages": messages, "temperature": 0, "max_tokens": max_tokens, "cache_prompt": True,
                "chat_template_kwargs": {"enable_thinking": False}}
        if schema:
            body["response_format"] = {"type": "json_schema", "json_schema": {"name": "incident_report", "schema": schema}}
        t0 = time.time()
        r = requests.post(self.url + "/v1/chat/completions", json=body, timeout=900)
        wall = time.time() - t0
        r.raise_for_status()
        j = r.json()
        content = j["choices"][0]["message"].get("content") or ""
        t, u = j.get("timings") or {}, j.get("usage") or {}
        return content, dict(wall_s=wall, prompt_tokens=u.get("prompt_tokens"), output_tokens=u.get("completion_tokens"),
                             prompt_ms=t.get("prompt_ms"), gen_ms=t.get("predicted_ms"), gen_tps=t.get("predicted_per_second"))


SYSTEM_PROMPT = f"""You turn a spoken incident report into a filled incident form.
Reply with ONLY one JSON object (no other text) with exactly these keys:
- "incident_type": one of {json.dumps(TYPES)}
- "severity": one of {json.dumps(SEVERITIES)}
- "location": where it happened, as the person said it, or null
- "date": "YYYY-MM-DD" or null
- "time": 24-hour "HH:MM" or null
- "persons_involved": list of people hurt or directly involved (names or roles), [] if none mentioned
- "equipment": equipment, vehicle or asset involved, or null
- "description": one or two sentences in English summarising what happened
- "immediate_actions": what was already done in response, or null
- "witnesses": list of people who saw it but were not involved, [] if none mentioned
- "follow_up": one short recommended next step

Rules:
1. Only use what the person said. If something was not mentioned, use null (or [] for lists). Never invent names, places or times.
2. incident_type, severity, description and follow_up must always be filled.
3. Times: convert to 24-hour time. "around 10 AM" -> "10:00". If AM/PM is not said, choose the most likely one; the incident has already happened. "just now", "5 minutes ago", "half an hour ago" -> work it out from the current time. If the person corrects themselves, use the corrected value.
4. Date: if a time or "today"/"this morning" is mentioned, the date is today; "yesterday" is the day before. If no time or day is mentioned at all, date and time are null.
5. Severity: Low = no injury, minor damage, under control. Medium = minor injury treated with first aid, or moderate damage or disruption. High = injury needing hospital treatment, significant damage, or danger to many people. Critical = life-threatening injury, fire or hazard still spreading, or a building being evacuated.
6. The person may speak Hinglish (Hindi in Roman letters mixed with English). Always write the form in English."""


def _nullable(t):
    return {"anyOf": [{"type": t}, {"type": "null"}]}


SCHEMA = {"type": "object", "additionalProperties": False, "required": EXPECTED_KEYS, "properties": {
    "incident_type": {"type": "string", "enum": TYPES}, "severity": {"type": "string", "enum": SEVERITIES},
    "location": _nullable("string"), "date": _nullable("string"), "time": _nullable("string"),
    "persons_involved": {"type": "array", "items": {"type": "string"}}, "equipment": _nullable("string"),
    "description": {"type": "string"}, "immediate_actions": _nullable("string"),
    "witnesses": {"type": "array", "items": {"type": "string"}}, "follow_up": {"type": "string"}}}


def user_message(text):
    return f'Current date and time: {NOW_TEXT}.\nWhat the person said:\n"""{text}"""'


def run_llm_on(server, items, mode, label):
    """items: list of (case, text the model sees)."""
    rows = []
    for c, inp in items:
        msgs = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_message(inp)}]
        try:
            raw, timing = server.chat(msgs, schema=SCHEMA if mode == "schema" else None)
        except Exception as e:
            raw, timing = "", {"wall_s": float("nan"), "error": repr(e)[:300]}
        s = score_case(c, raw)
        rows.append({"label": label, "mode": mode, "case": c["id"], "lang": c["lang"], **s, **timing, "input": inp, "raw": raw})
        print(f"  {label[:40]:40s} {mode:6s} {c['id']}  fields {s['fields_ok']}/{s['fields_n']}  "
              f"json={'ok' if s['json_valid'] else 'BAD'}  {timing.get('wall_s', float('nan')):.1f}s")
    return rows

In [ ]:
LLM_ROWS, LLM_FILES = [], {}
if RUN_LLM:
    from huggingface_hub import hf_hub_download
    LLAMA_EXE, LLAMA_ENV = get_llama_server()
    subprocess.run(["pkill", "-f", "llama-server"], capture_output=True)
    for name, cfg in LLM_CANDIDATES.items():
        if not cfg["run"]:
            continue
        print(f"\n=== {name} ===")
        srv = None
        try:
            repo, fname = find_gguf(cfg["repos"], GGUF_QUANTS)
            if not repo:
                print("  No 4-bit GGUF found in", cfg["repos"])
                continue
            print(f"  using {repo}/{fname}")
            path = hf_hub_download(repo, fname)
            srv = LlamaServer(LLAMA_EXE, LLAMA_ENV, path).start()
            LLM_FILES[name] = path
            print(f"  loaded in {srv.load_s:.0f}s, file {os.path.getsize(path) / 1e9:.2f} GB")
            srv.chat([{"role": "user", "content": "Say OK."}], max_tokens=5)   # warm-up
            for mode in LLM_MODES:
                rows = run_llm_on(srv, [(c, c["text"]) for c in CASES], mode, label=name)
                for r in rows:
                    r.update(gguf=f"{repo}/{fname}", size_gb=round(os.path.getsize(path) / 1e9, 2), load_s=srv.load_s)
                LLM_ROWS += rows
        except Exception as e:
            print(f"  {name} failed -> {repr(e)[:500]}")
        finally:
            if srv:
                srv.stop()
    if LLM_ROWS:
        LLM_DF = pd.DataFrame(LLM_ROWS)
        LLM_DF.to_csv(OUT / "llm_per_report.csv", index=False)
        LLM_SUMMARY = summarize_llm(LLM_DF)
        LLM_SUMMARY.to_csv(OUT / "llm_summary.csv", index=False)
        display(LLM_SUMMARY.T)
        print("\nWorst reports per model (fields correct out of 8):")
        display(LLM_DF.pivot_table(index="case", columns=["label", "mode"], values="fields_ok").T)

### Look at the actual answers
Numbers hide *why* a model fails. This shows, for each report, what each model put in the form.

In [ ]:
if LLM_ROWS:
    for cid in [c["id"] for c in CASES]:
        print(f"\n--- {cid}: {CASE_BY_ID[cid]['text']}")
        for r in LLM_DF[LLM_DF.case == cid].itertuples():
            print(f"  [{r.label} | {r.mode}] {r.fields_ok}/8 ->", (r.pred_json or repr(r.raw[:300])))

## Part 2: Text-to-speech (int8 ONNX on CPU via sherpa-onnx)

Each voice reads the English test reports (these recordings are reused as test audio in Part 3) and a
typical confirmation sentence. Measured:
* **RTF** (real-time factor) = seconds of computing per second of audio. Below 1 means faster than real time.
* **confirmation latency** = time to produce the whole confirmation sentence (non-streaming).
* **intelligibility** = Whisper-small's word error rate on this voice (filled in after Part 3; lower is clearer).
* Listen to the samples below each voice. Naturalness needs human ears.

Note: the MeloTTS build here is sherpa-onnx's Chinese+English version; AI Hub's MeloTTS-EN is English-only, so it may sound better.
Voices that ship without an int8 file are quantized here with ONNX Runtime's dynamic int8 quantization (shown in the table).

In [ ]:
def spoken(s):
    """Write numbers as words so every voice reads them the same way (also used to normalise transcripts)."""
    from num2words import num2words

    def _time(m):
        h, mi = int(m.group(1)), m.group(2)
        if mi == "00":
            return num2words(h)
        return f"{num2words(h)} {'oh ' + num2words(int(mi)) if mi.startswith('0') else num2words(int(mi))}"
    s = re.sub(r"\b(\d{1,2}):(\d{2})\b", _time, s)
    s = re.sub(r"\b(\d+)(st|nd|rd|th)\b", lambda m: num2words(int(m.group(1)), to="ordinal"), s)
    s = re.sub(r"\d+", lambda m: f" {num2words(int(m.group(0)))} ", s)
    return re.sub(r"\s+", " ", s).strip()


def wer_norm(s):
    return norm(spoken(norm(s)))


def to16k(y, sr):
    from scipy.signal import resample_poly
    y = np.asarray(y, dtype=np.float32)
    if y.ndim > 1:
        y = y.mean(axis=1)
    if sr != 16000:
        g = math.gcd(int(sr), 16000)
        y = resample_poly(y, 16000 // g, int(sr) // g).astype(np.float32)
    return y


def gh_release_assets(repo, tag):
    rel = requests.get(f"https://api.github.com/repos/{repo}/releases/tags/{tag}", timeout=60)
    rel.raise_for_status()
    rel = rel.json()
    assets, seen = list(rel.get("assets", [])), set()
    seen.update(a["name"] for a in assets)
    for page in range(1, 30):
        r = requests.get(f"https://api.github.com/repos/{repo}/releases/{rel['id']}/assets",
                         params={"per_page": 100, "page": page}, timeout=60)
        batch = r.json() if r.ok else []
        if not isinstance(batch, list) or not batch:
            break
        for a in batch:
            if a["name"] not in seen:
                assets.append(a)
                seen.add(a["name"])
    return {a["name"]: a["browser_download_url"] for a in assets}


def pick_asset(names, patterns):
    for p in patterns:
        hits = [n for n in names if n.startswith(p) and n.endswith(".tar.bz2")]
        if hits:
            return sorted(hits, key=lambda n: ("int8" not in n, len(n)))[0]
    return None


def fetch_tts_model(name, url, root):
    root.mkdir(parents=True, exist_ok=True)
    arch = root / name
    if not arch.exists():
        download(url, arch)
    with tarfile.open(arch, "r:bz2") as tf:
        top = {m.name.split("/")[0] for m in tf.getmembers() if m.name and not m.name.startswith(".")}
        tf.extractall(root)
    d = root / name[: -len(".tar.bz2")]
    return d if d.exists() else root / sorted(top)[0]


def _make_tts(d, model_file):
    import sherpa_onnx
    tokens, data, dct, voices = d / "tokens.txt", d / "espeak-ng-data", d / "dict", d / "voices.bin"
    lex = sorted(d.glob("lexicon*.txt"), key=lambda p: ("en" not in p.name, p.name))
    common = dict(num_threads=THREADS, provider="cpu", debug=False)
    if voices.exists():
        kc = dict(model=str(model_file), voices=str(voices), tokens=str(tokens), data_dir=str(data) if data.exists() else "")
        if dct.exists():
            kc["dict_dir"] = str(dct)
        if lex:
            kc["lexicon"] = ",".join(map(str, lex))
        mc = sherpa_onnx.OfflineTtsModelConfig(kokoro=sherpa_onnx.OfflineTtsKokoroModelConfig(**kc), **common)
    else:
        vc = dict(model=str(model_file), tokens=str(tokens), lexicon=str(lex[0]) if lex else "",
                  data_dir=str(data) if data.exists() else "", dict_dir=str(dct) if dct.exists() else "")
        mc = sherpa_onnx.OfflineTtsModelConfig(vits=sherpa_onnx.OfflineTtsVitsModelConfig(**vc), **common)
    cfg = sherpa_onnx.OfflineTtsConfig(model=mc, max_num_sentences=1)
    if not cfg.validate():
        raise ValueError("sherpa-onnx rejected the config for " + d.name)
    tts = sherpa_onnx.OfflineTts(cfg)
    test = tts.generate("Testing one two three.", sid=0, speed=1.0)
    if len(test.samples) == 0:
        raise ValueError("model produced no audio")
    return tts


def build_tts(d):
    """Uses the shipped int8 model if there is one, otherwise makes an int8 copy with ONNX Runtime."""
    onnx = sorted(d.glob("*.onnx"))
    shipped = [p for p in onnx if "int8" in p.name]
    if shipped:
        return _make_tts(d, shipped[0]), "int8 (shipped)", shipped[0]
    if not onnx:
        raise FileNotFoundError(f"no .onnx file in {d}")
    from onnxruntime.quantization import quantize_dynamic, QuantType
    src = onnx[0]
    for ops, label in ((["MatMul", "Gemm", "Conv"], "int8 dynamic, MatMul+Conv (made here)"),
                       (["MatMul", "Gemm"], "int8 dynamic, MatMul only (made here)")):
        dst = d / f"{src.stem}.dyn-int8-{len(ops)}ops.onnx"
        try:
            if not dst.exists():
                quantize_dynamic(str(src), str(dst), weight_type=QuantType.QUInt8, op_types_to_quantize=ops)
            return _make_tts(d, dst), label, dst
        except Exception as e:
            print(f"    {label} failed: {repr(e)[:200]}")
    raise RuntimeError("could not build a working int8 version")

In [ ]:
TTS_ROWS, TTS_AUDIO, TTS_SAMPLES = [], {}, {}
CONFIRM_TEXT = "Your incident report has been submitted. The maintenance team has been notified."
if RUN_TTS:
    import soundfile as sf
    from IPython.display import Audio
    try:
        TTS_ASSETS = gh_release_assets("k2-fsa/sherpa-onnx", "tts-models")
        print(f"{len(TTS_ASSETS)} sherpa-onnx voice downloads listed")
    except Exception as e:
        TTS_ASSETS = {}
        print("Could not list sherpa-onnx voices:", repr(e)[:300])
    for voice, patterns in TTS_CANDIDATES.items():
        print(f"\n=== {voice} ===")
        asset = pick_asset(TTS_ASSETS, patterns)
        if not asset:
            print("  no download matching", patterns)
            continue
        try:
            t0 = time.time()
            d = fetch_tts_model(asset, TTS_ASSETS[asset], WORK / "tts_models")
            tts, quant, model_file = build_tts(d)
            load_s = time.time() - t0
        except Exception as e:
            print(f"  {asset}: failed -> {repr(e)[:400]}")
            continue
        print(f"  {asset} | {quant} | {model_file.stat().st_size / 1e6:.0f} MB")
        out_dir = OUT / "audio" / re.sub(r"\W+", "_", voice)
        out_dir.mkdir(parents=True, exist_ok=True)
        voice_audio, synth_s, audio_s = {}, 0.0, 0.0
        try:
            for c in EN_CASES:
                t0 = time.time()
                a = tts.generate(spoken(c["text"]), sid=0, speed=1.0)
                dt = time.time() - t0
                y = np.asarray(a.samples, dtype=np.float32)
                dur = len(y) / a.sample_rate
                sf.write(out_dir / f"{c['id']}.wav", y, a.sample_rate)
                voice_audio[c["id"]] = (to16k(y, a.sample_rate), dur)
                synth_s, audio_s = synth_s + dt, audio_s + dur
            t0 = time.time()
            a = tts.generate(CONFIRM_TEXT, sid=0, speed=1.0)
            confirm_s = time.time() - t0
        except Exception as e:
            print(f"  {voice} failed while speaking -> {repr(e)[:400]}")
            continue
        TTS_AUDIO[voice] = voice_audio
        TTS_SAMPLES[voice] = (np.asarray(a.samples, dtype=np.float32), a.sample_rate)
        sf.write(out_dir / "confirmation.wav", TTS_SAMPLES[voice][0], a.sample_rate)
        TTS_ROWS.append({"voice": voice, "download": asset, "quantization": quant,
                         "model MB": round(model_file.stat().st_size / 1e6), "sample rate": a.sample_rate,
                         "load s": round(load_s, 1), "RTF": round(synth_s / max(audio_s, 1e-9), 3),
                         "confirmation latency s": round(confirm_s, 2), "audio s generated": round(audio_s, 1)})
        print(f"  RTF {synth_s / max(audio_s, 1e-9):.3f}, confirmation took {confirm_s:.2f}s")
        display(Audio(TTS_SAMPLES[voice][0], rate=a.sample_rate))
        del tts
    if TTS_ROWS:
        TTS_DF = pd.DataFrame(TTS_ROWS)
        display(TTS_DF)

## Part 3: Speech-to-text (Whisper int8 on CPU via faster-whisper)

Test audio:
* every TTS voice reading the English reports, **clean** and with **background chatter** at 10 dB (a busy hall);
* **Svarah**: real Indian-accented English speech (optional, needs the HF token);
* **your own recordings** (optional; best test of all, see `OWN_AUDIO_DIR`).

**WER** = word error rate (lower is better; 0.10 means 1 word in 10 is wrong). **CER** = the same per character.
**RTF** = seconds of computing per second of audio.

In [ ]:
def add_chatter(clean, others, snr_db, k=3, seed=0):
    rng = random.Random(seed)
    picks = rng.sample(others, min(k, len(others)))
    babble = np.zeros_like(clean)
    for o in picks:
        if len(o) == 0:
            continue
        babble += np.tile(o, int(np.ceil(len(clean) / len(o))))[: len(clean)]
    pc, pb = float(np.mean(clean ** 2)) + 1e-12, float(np.mean(babble ** 2)) + 1e-12
    mix = clean + babble * math.sqrt(pc / (pb * 10 ** (snr_db / 10)))
    return (mix / max(1.0, float(np.max(np.abs(mix))))).astype(np.float32)


def load_svarah(n):
    try:
        import soundfile as sf
        from datasets import load_dataset, get_dataset_split_names, Audio as HFAudio
        tok = get_hf_token()
        split = get_dataset_split_names("ai4bharat/Svarah", token=tok)[0]
        ds = load_dataset("ai4bharat/Svarah", split=split, streaming=True, token=tok)
        feats = ds.features or {}
        acol = next((k for k, v in feats.items() if isinstance(v, HFAudio)), "audio")
        ds = ds.cast_column(acol, HFAudio(decode=False)).shuffle(seed=0, buffer_size=300)
        items = []
        for i, ex in enumerate(ds):
            if len(items) >= n:
                break
            a = ex.get(acol)
            data = a.get("bytes") if isinstance(a, dict) else None
            text = next((ex[k] for k in ("text", "transcript", "transcription", "sentence", "normalized_text")
                         if isinstance(ex.get(k), str) and ex[k].strip()), None)
            if not data or not text:
                continue
            y, sr = sf.read(io.BytesIO(data), dtype="float32")
            y = to16k(y, sr)
            items.append(dict(id=f"svarah{i}", case_id=None, audio=y, dur=len(y) / 16000, ref=text, lang="en"))
        print(f"Svarah: {len(items)} clips")
        return items
    except Exception as e:
        print("Svarah skipped:", repr(e)[:300])
        print("  (Add an HF_TOKEN secret and accept the dataset terms on Hugging Face to include it.)")
        return []


def load_own(folder):
    if not folder:
        return []
    import soundfile as sf
    folder = Path(folder)
    meta = pd.read_csv(folder / "metadata.csv")
    items = []
    for i, r in meta.iterrows():
        p = folder / str(r["file"])
        try:
            y, sr = sf.read(p, dtype="float32")
        except Exception:
            import librosa
            y, sr = librosa.load(p, sr=None)
        y = to16k(y, sr)
        cid = r.get("case_id") if "case_id" in meta.columns and isinstance(r.get("case_id"), str) else None
        lang = r.get("lang") if "lang" in meta.columns and isinstance(r.get("lang"), str) else "en"
        items.append(dict(id=f"own{i}", case_id=cid, audio=y, dur=len(y) / 16000, ref=str(r["text"]), lang=lang))
    print(f"Own recordings: {len(items)} files")
    return items


ASR_ITEMS = {}
for voice, d in TTS_AUDIO.items():
    clean = [dict(id=cid, case_id=cid, audio=a, dur=dur, ref=CASE_BY_ID[cid]["text"], lang="en") for cid, (a, dur) in d.items()]
    ASR_ITEMS[f"TTS {voice} | clean"] = clean
    ASR_ITEMS[f"TTS {voice} | chatter {NOISE_SNR_DB}dB"] = [
        dict(it, audio=add_chatter(it["audio"], [o["audio"] for o in clean if o["id"] != it["id"]], NOISE_SNR_DB, seed=j))
        for j, it in enumerate(clean)]
if RUN_ASR and USE_SVARAH:
    sv = load_svarah(SVARAH_N)
    if sv:
        ASR_ITEMS["Svarah (Indian English)"] = sv
own = load_own(OWN_AUDIO_DIR)
if own:
    ASR_ITEMS["Own recordings"] = own
print({k: len(v) for k, v in ASR_ITEMS.items()})

In [ ]:
ASR_ROWS, TRANSCRIPTS, ASR_TIME = [], {}, {}
if RUN_ASR and ASR_ITEMS:
    import jiwer
    from faster_whisper import WhisperModel
    for size in ASR_MODELS:
        print(f"\n=== whisper-{size} int8 ===")
        t0 = time.time()
        model = WhisperModel(size, device="cpu", compute_type="int8", cpu_threads=THREADS)
        load_s = time.time() - t0
        for set_name, items in ASR_ITEMS.items():
            refs, hyps, proc, dur = [], [], 0.0, 0.0
            for it in items:
                t0 = time.time()
                segs, _ = model.transcribe(it["audio"], language=it.get("lang") if it.get("lang") in ("en", "hi") else None,
                                           beam_size=ASR_BEAM, condition_on_previous_text=False, vad_filter=False)
                hyp = " ".join(s.text.strip() for s in segs).strip()
                dt = time.time() - t0
                TRANSCRIPTS[(size, set_name, it["id"])], ASR_TIME[(size, set_name, it["id"])] = hyp, dt
                r = wer_norm(it["ref"])
                if r:
                    refs.append(r)
                    hyps.append(wer_norm(hyp))
                proc, dur = proc + dt, dur + it["dur"]
            row = dict(model=f"whisper-{size} int8", set=set_name, clips=len(items),
                       WER=round(jiwer.wer(refs, hyps), 3), CER=round(jiwer.cer(refs, hyps), 3),
                       RTF=round(proc / max(dur, 1e-9), 3), **{"load s": round(load_s, 1)})
            ASR_ROWS.append(row)
            print(f"  {set_name:45s} WER {row['WER']:.3f}  CER {row['CER']:.3f}  RTF {row['RTF']:.3f}")
        del model
    ASR_DF = pd.DataFrame(ASR_ROWS)
    ASR_DF.to_csv(OUT / "asr_summary.csv", index=False)
    print("\nWER (lower is better):")
    display(ASR_DF.pivot_table(index="set", columns="model", values="WER"))
    print("RTF (seconds of compute per second of audio):")
    display(ASR_DF.pivot_table(index="set", columns="model", values="RTF"))
    # one example so you can see the kind of mistakes
    k = next(iter(ASR_ITEMS))
    it = ASR_ITEMS[k][0]
    print("\nExample:", k, "\n  said :", it["ref"], "\n  heard:", TRANSCRIPTS[(ASR_MODELS[-1], k, it["id"])])

if TTS_ROWS and ASR_ROWS:
    intel = {r["set"].split(" | ")[0][4:]: r["WER"] for r in ASR_ROWS
             if r["model"] == "whisper-small int8" and r["set"].endswith("| clean")}
    TTS_DF["intelligibility WER (whisper-small)"] = TTS_DF["voice"].map(intel)
    TTS_DF.to_csv(OUT / "tts_summary.csv", index=False)
    display(TTS_DF)

## Part 4: Full pipeline, speech in → form out

The best LLM from Part 1 now fills the form from **Whisper's transcript** instead of the typed text,
using the clearest TTS voice (clean and with chatter) and your own recordings if they have a `case_id`.
Comparing with the typed-text score shows how much speech-recognition mistakes cost.

In [ ]:
PIPE_ROWS = []
if RUN_PIPELINE and LLM_ROWS and TRANSCRIPTS:
    base = LLM_DF[LLM_DF["mode"] == "free"].copy()
    base["acc"] = base[FIELD_COLS].astype(float).mean(axis=1)
    best_llm = base.groupby("label")["acc"].mean().idxmax()
    clean_sets = [r for r in ASR_ROWS if r["model"] == f"whisper-{PIPELINE_ASR} int8" and r["set"].endswith("| clean")]
    sets = []
    if clean_sets:
        best_voice_set = min(clean_sets, key=lambda r: r["WER"])["set"]
        sets += [best_voice_set, best_voice_set.replace("| clean", f"| chatter {NOISE_SNR_DB}dB")]
    if "Own recordings" in ASR_ITEMS:
        sets.append("Own recordings")
    print("LLM:", best_llm, "| ASR: whisper-" + PIPELINE_ASR, "| audio sets:", sets)
    srv = LlamaServer(LLAMA_EXE, LLAMA_ENV, LLM_FILES[best_llm]).start()
    try:
        for set_name in sets:
            items = [it for it in ASR_ITEMS.get(set_name, []) if it.get("case_id") in CASE_BY_ID
                     and CASE_BY_ID[it["case_id"]] in CASES and (PIPELINE_ASR, set_name, it["id"]) in TRANSCRIPTS]
            pairs = [(CASE_BY_ID[it["case_id"]], TRANSCRIPTS[(PIPELINE_ASR, set_name, it["id"])]) for it in items]
            rows = run_llm_on(srv, pairs, "free", label=f"speech: {set_name}")
            for r, it in zip(rows, items):
                r["asr_s"] = ASR_TIME[(PIPELINE_ASR, set_name, it["id"])]
                r["total_s"] = r["asr_s"] + (r.get("wall_s") or 0)
            PIPE_ROWS += rows
    finally:
        srv.stop()
    if PIPE_ROWS:
        PIPE_DF = pd.DataFrame(PIPE_ROWS)
        PIPE_DF.to_csv(OUT / "pipeline_per_report.csv", index=False)
        ids = set(PIPE_DF["case"])
        typed = base[(base["label"] == best_llm) & (base["case"].isin(ids))].assign(label=f"typed text ({best_llm})")
        PIPE_SUMMARY = summarize_llm(pd.concat([typed, PIPE_DF], ignore_index=True))
        PIPE_SUMMARY["speech→form sec"] = [float("nan")] + [PIPE_DF[PIPE_DF.label == l]["total_s"].mean()
                                                           for l in PIPE_SUMMARY["model"][1:]]
        PIPE_SUMMARY.to_csv(OUT / "pipeline_summary.csv", index=False)
        display(PIPE_SUMMARY[["model", "reports", "field accuracy %", "made-up values %", "right follow-up decision %",
                              "type %", "location %", "time %", "speech→form sec"]].round(1))

## Summary

In [ ]:
print("=" * 100)
if LLM_ROWS:
    cols = ["model", "mode", "field accuracy %", "valid JSON %", "made-up values %", "invented people",
            "right follow-up decision %", "severity ±1 %", "sec / report"]
    print("LLM form filling (typed text):")
    display(LLM_SUMMARY[cols])
if TTS_ROWS:
    print("Text-to-speech:")
    display(TTS_DF)
if ASR_ROWS:
    print("Speech-to-text, WER by test set:")
    display(ASR_DF.pivot_table(index="model", columns="set", values="WER"))
if PIPE_ROWS:
    print("Speech in -> form out:")
    display(PIPE_SUMMARY[["model", "field accuracy %", "made-up values %", "right follow-up decision %", "speech→form sec"]])
print(f"\nAll tables, per-report answers and audio are in {OUT}")